In [ ]:
# Example 1: gRPC status codes enum + constants surface
from grpclib.const import Status, Cardinality

# Status is an IntEnum mirroring the gRPC status codes
assert Status.OK.value == 0
assert Status.CANCELLED.value == 1
assert Status.NOT_FOUND.value == 5
assert Status.UNAUTHENTICATED.value == 16

# Cardinality enumerates the 4 RPC types
card_names = {c.name for c in Cardinality}
assert {'UNARY_UNARY', 'UNARY_STREAM', 'STREAM_UNARY', 'STREAM_STREAM'} <= card_names

print(f"Status codes: {len(list(Status))} entries")
print(f"Cardinalities: {card_names}")

In [ ]:
# Example 2: GRPCError exception construction + attributes
from grpclib.exceptions import GRPCError, ProtocolError, StreamTerminatedError
from grpclib.const import Status

err = GRPCError(Status.NOT_FOUND, message='resource not found', details=None)
assert err.status == Status.NOT_FOUND
assert err.message == 'resource not found'
assert isinstance(err, Exception)

# Subclass hierarchy
assert issubclass(ProtocolError, Exception)
assert issubclass(StreamTerminatedError, Exception)

print(f"GRPCError(NOT_FOUND): status={err.status.name} message={err.message!r}")

In [ ]:
# Example 3: Server class construction (no bind, just object init)
from grpclib.server import Server
from grpclib.health.service import Health

# Health service ships with grpclib for gRPC health-checking protocol
health_svc = Health()
server = Server([health_svc])

# Server exposes __mapping__ via its handlers; just verify object init
assert server is not None
assert hasattr(server, 'start')
assert hasattr(server, 'close')
assert hasattr(server, 'wait_closed')

print(f"Server constructed with {len([health_svc])} handler(s)")
print(f"Health service type: {type(health_svc).__name__}")

In [ ]:
# Example 4: Channel client construction (no connect)
from grpclib.client import Channel

# Channel is the client-side connection holder; ctor is sync, no I/O
channel = Channel(host='127.0.0.1', port=50051)

assert channel is not None
assert hasattr(channel, 'request')
assert hasattr(channel, 'close')

# Close it cleanly (sync close on un-opened channel is a no-op)
channel.close()
print(f"Channel constructed and closed cleanly: {type(channel).__name__}")

In [ ]:
# Example 5: End-to-end in-process server <-> client round-trip via health check
import asyncio
from grpclib.server import Server
from grpclib.client import Channel
from grpclib.health.service import Health
from grpclib.health.v1.health_pb2 import HealthCheckRequest
from grpclib.health.v1.health_grpc import HealthStub

health_svc = Health()
server = Server([health_svc])
await server.start('127.0.0.1', 0)
port = server._server.sockets[0].getsockname()[1]

channel = Channel(host='127.0.0.1', port=port)
stub = HealthStub(channel)
reply = await stub.Check(HealthCheckRequest(service=''))

channel.close()
server.close()
await server.wait_closed()

assert reply is not None
print(f"Health check reply status: {reply.status}")
assert reply.status == 1  # SERVING